### Homework 4 - Camera Calibration

Student = David Medina

Id = F11115117

In [1]:
# libraries
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt

In [2]:
# to limit the digits displayed per number
np.set_printoptions(suppress=True, precision=4)

In [3]:
# # read img
# img = cv.imread('imgToCalib.png') # opencv works with BGR
# img_RGB = cv.cvtColor(img, cv.COLOR_RGB2BGR) # matplotlib works with RGB

# plt.imshow(img_RGB)
# plt.show()

In [4]:
# points_2d = np.array([[1049, 58],
#                       [1440, 175],
#                       [631, 348],
#                       [358, 160],
#                       [1376, 737],
#                       [686, 1024],
#                       [437, 710]])

In [5]:
# # draw circles of the selected points for reference
# img2save = img_RGB.copy()

# for i, point in enumerate(points_2d, start=1):
#     x, y = point
#     cv.circle(img2save, (x, y), 1, (0, 255, 255), -1) # points
#     cv.putText(img2save, f'({x}, {y})', (x - 40, y - 30), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2) # coordinates
#     # cv.putText(img_spare, str(i), (x - 40, y - 30), cv.FONT_HERSHEY_SIMPLEX, 3, (0, 255, 255), 10) # labels

# # show image with reference points
# plt.imshow(img2save)
# plt.show()

# # save image with reference points
# img2save = cv.cvtColor(img2save, cv.COLOR_RGB2BGR)
# cv.imwrite('reference_points.png', img2save)

### Calculate Intrinsic Parameter K

In [6]:
# homography for upper square                      

upper_square = np.array([[1049, 58],
                         [358, 160],
                         [631, 348],
                         [1440, 175]])

real_dimension_8x6 = np.array([[0, 0],
                               [8, 0], 
                               [8, 6],
                               [0, 6]])

h_upper, _ = cv.findHomography(srcPoints=real_dimension_8x6, dstPoints=upper_square)

h_upper_t = h_upper.T
h_upper_t

array([[ -94.6876,    9.0349,   -0.0232],
       [  15.1163,   13.4175,   -0.0348],
       [1049.    ,   58.    ,    1.    ]])

In [7]:
# homography for right square

right_square = np.array([[1376, 737],
                         [1440, 175],
                         [631, 348],
                         [686, 1024]])

real_dimension_8x6 = np.array([[0, 0],
                               [0, 6],
                               [8, 6], 
                               [8, 0]])

h_right, _ = cv.findHomography(srcPoints=real_dimension_8x6, dstPoints=right_square)

h_right_t = h_right.T
h_right_t

array([[-103.6363,    9.9223,   -0.0253],
       [ -21.3279,  -97.5549,   -0.0222],
       [1376.    ,  737.    ,    1.    ]])

In [8]:
# homography for left square

left_square = np.array([[437, 710], 
                        [686, 1024],
                        [631, 348],
                        [358, 160]])

real_dimension_6x6 = np.array([[0, 0], 
                               [0, 6],
                               [6, 6],
                               [6, 0]])

h_left, _ = cv.findHomography(srcPoints=real_dimension_6x6, dstPoints=left_square)

h_left_t = h_left.T
h_left_t

array([[-21.097 , -95.2109,  -0.0222],
       [ 15.9632,  14.2143,  -0.0372],
       [437.    , 710.    ,   1.    ]])

In [9]:
# based on Lecture 07, page 50 formula
def calculate_matrix_A(h_1, h_2, h_3):
    A = []
    
    for h in [h_1, h_2, h_3]:
        A.append([h[0,0]*h[1,0], h[0,0]*h[1,1] + h[0,1]*h[1,0], h[0,0]*h[1,2] + h[0,2]*h[1,0], h[0,1]*h[1,1], h[0,1]*h[1,2] + h[0,2]*h[1,1], h[0,2]*h[1,2]])
        A.append([h[0,0]**2 - h[1,0]**2, 2*(h[0,0]*h[0,1] - h[1,0]*h[1,1]), 2*(h[0,0]*h[0,2] - h[1,0]*h[1,2]), h[0,1]**2 - h[1,1]**2, 2*(h[0,1]*h[0,2] - h[1,1]*h[1,2]), h[0,2]**2 - h[1,2]**2])
    
    return np.array(A)

In [10]:
matrix_A = calculate_matrix_A(h_upper_t, h_right_t, h_left_t)
matrix_A

array([[-1431.33  , -1133.8955,     2.9401,   121.2256,    -0.6256,
            0.0008],
       [ 8737.231 , -2116.6299,     5.448 ,   -98.4   ,     0.5131,
           -0.0007],
       [ 2210.3473,  9898.606 ,     2.8432,  -967.9676,     2.252 ,
            0.0006],
       [10285.6003, -6217.9054,     4.3055, -9418.5065,    -4.838 ,
            0.0001],
       [ -336.7757, -1819.7526,     0.4317, -1353.3571,     3.2294,
            0.0008],
       [  190.2574,  3563.5119,     2.1231,  8863.0758,     5.2764,
           -0.0009]])

In [11]:
# find through IAC
def calculate_w(A):
    w = []

    # SVD to find v
    u, s, v = np.linalg.svd(A)
    v = np.transpose(v)

    w = np.array([[v[0,5], v[1,5], v[2,5]],
                  [v[1,5], v[3,5], v[4,5]],
                  [v[2,5], v[4,5], v[5,5]]], dtype=np.float64)
    
    return np.array(w)

In [12]:
matrix_w = calculate_w(matrix_A)
matrix_w

array([[ 0.    ,  0.    , -0.0002],
       [ 0.    ,  0.    , -0.0001],
       [-0.0002, -0.0001,  1.    ]])

In [13]:
def calculate_K(w):
    K = []

    inv_w = np.linalg.inv(w)
    inv_w = inv_w / inv_w[2,2]

    c = inv_w[0,2]
    e = inv_w[1,2]
    d = sqrt(inv_w[1,1] - e**2)
    b = (inv_w[0,1] - c*e) / d
    a = sqrt(inv_w[0,0] - b**2 - c**2)

    K = np.array([[a, b, c],
                  [0, d, e],
                  [0, 0, 1]], dtype=np.float64)
    
    return K

In [14]:
matrix_K = calculate_K(matrix_w)

print('Intrinsic parameter K:')
print(matrix_K)

Intrinsic parameter K:
[[1868.8761  -11.206   953.4564]
 [   0.     1865.8248  535.4951]
 [   0.        0.        1.    ]]


In [15]:
#   K matrix - GROUND TRUTH
#
#   [1866.7,  0.00,    960.0
#    0.00,    1866.7,  540.0
#    0.00,    0.00,    1.0]

### Calculate Extrinsic Parameter Rt

In [16]:
# from Zhang's method shown in Lecture 07, page 62
def calculate_Rt(h, K):
    Rt = []

    r = np.linalg.inv(K) @ h
    
    lenght_r1 = np.linalg.norm(r[:,0])
    
    r1 = r[:, 0] / lenght_r1 # normalized
    r2 = r[:, 1] / lenght_r1 # not unit lenght
    
    r3 = np.cross(r1, r2) # not unit lenght
    lenght_r3 = np.linalg.norm(r3)
    r3 = r3 / lenght_r3 # normalized

    r2 = np.cross(r3, r1) # unit lenght

    t = r[:, 2] / lenght_r1
    
    Rt = np.column_stack([r1, r2, r3, t])
    
    return Rt

In [17]:
matrix_Rt = calculate_Rt(h_upper, matrix_K)
matrix_Rt

array([[-0.8313,  0.5559, -0.0006,  1.0638],
       [ 0.2468,  0.3681, -0.8964, -5.4898],
       [-0.4981, -0.7453, -0.4432, 21.4514]])

In [18]:
displace_to_origin = np.array([[1, 0, 0, 0],
                               [0, 1, 0, 0],
                               [0, 0, 1, -6],
                               [0, 0, 0, 1]])

Rt_at_origin = matrix_Rt @ displace_to_origin

print('Extrinsic parameter Rt at origin:')
print(Rt_at_origin)

Extrinsic parameter Rt at origin:
[[-0.8313  0.5559 -0.0006  1.0674]
 [ 0.2468  0.3681 -0.8964 -0.1112]
 [-0.4981 -0.7453 -0.4432 24.1106]]


In [19]:
#   RT matrix - GROUND TRUTH (updated @2024/05/20)
#
#   [ -0.8309382  0.5563648   0.00000     0.4801183
#     0.2477334   0.369993    -0.8953956  -0.0613666
#     -0.4981667  -0.7440184  -0.4452714  11.221666 ]

### Calculate Camera Position

In [20]:
# rotation matrix R 
R = Rt_at_origin[:, 0:3]
R

array([[-0.8313,  0.5559, -0.0006],
       [ 0.2468,  0.3681, -0.8964],
       [-0.4981, -0.7453, -0.4432]])

In [21]:
# translation vector t
t = Rt_at_origin[:, 3]
t

array([ 1.0674, -0.1112, 24.1106])

In [22]:
# camera position in the world cordinate system
camera_position = -(R.T @ t)
camera_position

array([12.924 , 17.4174, 10.5867])